# P1｜AST Joint-Native Reference

**状态：Design / Not Ready；8/20 core。** 目的：建立四数据集 joint-native 参考包。ICBHI、SPRSound、KAUH 使用 AST pooled/native lanes；HF 暂时保持固定 native temporal reference。P1 不支持“四数据集共享 AST encoder”结论。

## 唯一变量、匹配对照与四数据集边界

P1 是 reference，不引入 eligibility、balanced sampler、PAFA、MVST、LODO 或共享 HF encoder。首轮 scope 固定为 frozen pretrained AST encoder + trainable shared projector 768→256 + dataset-native heads；HF fixed temporal reference 独立，不进入 shared projector。sampler 固定为 source-proportional。

single-source comparator 使用相同架构的 projector + 对应 native head；joint 条件的唯一结构性差异是同一个 shared projector 接收 ICBHI、SPRSound、KAUH 三个 lanes。后续 P2–P5 必须匹配 HF reference、split、heads、sampler、scope、update budget 与 selection。

- ICBHI：cycle → flat4 [B,4]；SPRSound：event → binary [B,2] 与 raw7 [B,7]；KAUH：recording → raw9 [B,9]，B/D/E 同 patient group。
- HF：15 秒 native temporal reference；missing/unknown/gap 不是 negative。不得把 HF reference 写成 AST shared lane。
- 输入/输出：16 kHz 音频与 immutable manifests → non-HF AST pooled [B,768] → shared projected [B,256] → dataset-native logits；HF 另行输出 fixed native time-aligned logits。各任务分开计分。

In [ ]:
from pathlib import Path
import os

PIPELINE = {
    "id": "P1",
    "seed": 20260728,
    "split_policy": "immutable_prerequisite_receipts",
    "non_hf_encoder": "AST_AudioSet_reference",
    "encoder_scope": "frozen_pretrained_AST",
    "shared_projector": "minimal_linear_projector_768_to_256",
    "shared_projector_bias": True,
    "shared_projector_lanes": ["ICBHI", "SPRSound", "KAUH"],
    "sampler": "source_proportional",
    "hf_lane": "fixed_native_temporal_reference",
    "hf_uses_shared_projector": False,
    "trainable_scope": "shared_projector_768_to_256_plus_dataset_native_heads",
    "contract_modules": ["baseline.multidataset_pipeline.contracts", "baseline.multidataset_pipeline.joint_native"],
    "engineering_test": "tests/test_multidataset_pipeline.py::JointNativeContractTest",
    "update_budget": None,
    "selection": None,
    "output_dir": "result/reproduce/P1_joint_native_reference",
    "receipt_path": "result/reproduce/P1_joint_native_reference/P1_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P1_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

必须先锁定四数据集 receipt/hash、原冻结 split/grouping、AST checkpoint provenance/input adapter、HF native comparator provenance、single-source 同架构 projector+head receipts、固定 trainable scope/source-proportional sampler、单一 update budget、validation-only selection 与独立 verifier。任何缺失均 fail closed；outer/test 在 selection 冻结前不可读。完整训练或服务器运行需另行批准。

In [ ]:
required = [PIPELINE["update_budget"], PIPELINE["selection"], APPROVAL_RECEIPT]
if any(value in (None, "") for value in required):
    raise RuntimeError("P1 fail closed: budget, selection, and approval receipt must be frozen")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "approval": str(approval_path), "execute": False}


## 输出、receipt 与结果表

receipt 至少记录 pipeline_id、config/manifest/checkpoint hashes、lane-wise prediction units、split/group hashes、seed、updates、trainable parameters、selection、native metrics、warnings 与 verifier status。

| Dataset / native task | Metric | Result | Decision |
|---|---:|---:|---|
| ICBHI / SPRSound / HF / KAUH | 待冻结 | Not run | 未判定 |

**Test Result = Not run。Decision = Not made。Claim boundary：只比较 non-HF 三 lane 的 joint shared-projector reference；不是覆盖 HF 的共享 AST encoder 证据。**